In [102]:
# RAG answer: before prompt ---> after prompt
# Before prompt: user query ---> query rewriting ---> query expansion ---> merge chunks ---> reranking ---> RankOrder validation class ---> top-s chunks
# Query expansion for both original quesion and rewritten question: query embedding ---> retriever ---> top-k chunks
# After prompt: prompt ---> LLM ---> answer
import os
import sys
from openai import OpenAI
from dotenv import load_dotenv
from huggingface_hub import login
from pydantic import BaseModel, Field
from pathlib import Path
from chromadb import PersistentClient
from IPython.display import display, Markdown
from litellm import completion
from tenacity import retry, wait_exponential, stop_after_attempt
# Add the utils folder to Python path
notebook_dir = os.path.dirname(os.path.abspath("__file__"))
pricer_path = os.path.join(notebook_dir, "../utils")
sys.path.insert(0, pricer_path)
# Class Item to validate the returned data and has lots of helper functions
from pricer.items import Item

In [103]:
# Load environment variables in a file called .env
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
# Check the keys
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")  

OpenAI API Key exists and begins sk-proj-


In [104]:
# Connect to OpenAI client library
openai = OpenAI()

In [105]:
# Login to Hugging Face
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [106]:
# Download the batches dataset from Hugging Face 
# It's only testing resource
username = "omarshaarawy11"
dataset = f"{username}/best_deal_03_batches"
train, val, test = Item.from_hub(dataset)
items = test
print(f"Loaded {len(items):,} items for flow testing")

Loaded 2,191 items for flow testing


In [107]:
# Configuraions
PROJECT_ROOT = Path(notebook_dir).parent.parent 
DB_NAME = str(PROJECT_ROOT / "notebooks" / "01_data_pipeline" / "preprocessed_db")
collection_name = "docs"
embedding_model = "text-embedding-3-large"
GENERATOR_MODEL = "gpt-4.1-mini"
wait = wait_exponential(multiplier=1, min=2, max=30) 
# Number of CPU processors
WORKERS = 3
# Number of chunks before reranking
RETRIEVAL_K = 10
# Number of chunks after reranking
RETRIEVAL_S = 10

SYSTEM_PROMPT = """
You are a pricing assistant for an e-commerce platform.

Your task is to estimate the price of the target product using
the retrieved products and their prices as reference examples.

Use the retrieved products that are most similar to the target product,
even if they are not an exact match - same category, similar brand tier,
similar features/size are enough to use as comparables. Reason about how
the target differs (bigger/smaller, more/fewer features, premium/budget
brand) and adjust your estimate accordingly.

Only return 0.00 if the retrieved context is from a clearly different
category or is otherwise unusable for comparison - not merely because no
retrieved product is an exact match.

Return ONLY one numeric value representing the estimated price
in US dollars. Do not include currency symbols, explanations, or
additional text.

Knowledge base context:
{context}
"""
print("Vector DB path:", DB_NAME)

Vector DB path: c:\Users\EGShaaraOm\OneDrive - NESTLE\Attachments\Work\Software Solutions\Projects\LLMs_Projects\Best_Deal_Gen_AI\notebooks\01_data_pipeline\preprocessed_db


In [108]:
# Initialize vector database
chroma_client = PersistentClient(path=DB_NAME)
collection = chroma_client.get_or_create_collection(name=collection_name)

In [109]:
# Invitisgate the vector database
print(collection.count())
print(collection.peek(limit=1))

16800
{'ids': ['0'], 'embeddings': array([[-0.01412964, -0.00593185, -0.02098083, ..., -0.00398636,
        -0.00984192,  0.00915527]]), 'documents': ['2 Pack Air Filter Factory Compatible Replacement for Broan Nutone S99010430-002, 99010430-002, 4512880 Range Hood Aluminum Grease Filters\n\nTitle: 2 Pack Air Filter Factory Compatible Replacement for Broan Nutone S99010430-002, 99010430-002, 4512880 Range Hood Aluminum Grease Filters\n\nCategory: appliances\n\nDescription: Title: 2-Pack Aluminum Range Hood Grease Filters  \nCategory: Kitchen Appliances  \nBrand: Air Filter Factory  \nDescription: Replacement aluminum grease filters compatible with Broan Nutone range hoods.  \nDetails: Made in the USA, these mesh filters are designed to capture grease and mist, ensuring clean operation.'], 'uris': None, 'included': ['metadatas', 'documents', 'embeddings'], 'data': None, 'metadatas': [{'price': 47.97, 'type': 'appliances', 'title': '2 Pack Air Filter Factory Compatible Replacement for Br

In [110]:
# Validation classes
# Result class
class Result(BaseModel):
    page_content: str
    metadata: dict

In [111]:
class RankOrder(BaseModel):
    order: list[int] = Field(
        description="The order of relevance of chunks, from most relevant to least relevant, by chunk id number"
    )

In [112]:
# Load the persisted BM25 sparse index (built once in 05_RAG_ingestion.ipynb).
import re
import pickle
from rank_bm25 import BM25Okapi

BM25_PATH = Path(DB_NAME) / "bm25_index.pkl"

def tokenize(text: str) -> list[str]:
    return re.findall(r"[a-z0-9]+", text.lower())

if BM25_PATH.exists():
    with open(BM25_PATH, "rb") as f:
        bm25_payload = pickle.load(f)
    bm25_index = BM25Okapi(bm25_payload["corpus_tokens"])
    print(f"BM25 index loaded: {len(bm25_payload['ids']):,} documents")
else:
    bm25_index = None
    bm25_payload = None
    print(f"WARNING: {BM25_PATH} not found - run 05_RAG_ingestion.ipynb first. "
          f"Falling back to dense-only retrieval.")


BM25 index loaded: 16,800 documents


In [113]:
# Reciprocal Rank Fusion 
def reciprocal_rank_fusion(rank_lists: list[list[str]], k_rrf: int = 60) -> list[str]:
    scores: dict[str, float] = {}
    for rank_list in rank_lists:
        for rank, doc_id in enumerate(rank_list, start=1):
            scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (k_rrf + rank)
    return [doc_id for doc_id, _ in sorted(scores.items(), key=lambda x: x[1], reverse=True)]


def bm25_search(question: str, k: int = RETRIEVAL_K):
    """Returns (ranked_ids, id -> (page_content, metadata) lookup) for a BM25 query."""
    if bm25_index is None:
        return [], {}
    tokens = tokenize(question)
    scores = bm25_index.get_scores(tokens)
    top_idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]
    ids = [bm25_payload["ids"][i] for i in top_idx]
    lookup = {
        bm25_payload["ids"][i]: (bm25_payload["documents"][i], bm25_payload["metadatas"][i])
        for i in top_idx
    }
    return ids, lookup


def fetch_context_hybrid(question: str, k: int = RETRIEVAL_K) -> list[Result]:
    """Dense (Chroma) + sparse (BM25) retrieval, fused with RRF."""
    # Dense
    query_vec = openai.embeddings.create(model=embedding_model, input=[question]).data[0].embedding
    dense_results = collection.query(query_embeddings=[query_vec], n_results=k)
    dense_ids = dense_results["ids"][0]
    dense_lookup = {
        doc_id: (doc, meta)
        for doc_id, doc, meta in zip(dense_ids, dense_results["documents"][0], dense_results["metadatas"][0])
    }

    # Sparse
    bm25_ids, bm25_lookup = bm25_search(question, k)

    # Fuse
    fused_ids = reciprocal_rank_fusion([dense_ids, bm25_ids])[:k]

    chunks = []
    for doc_id in fused_ids:
        doc, meta = dense_lookup.get(doc_id) or bm25_lookup.get(doc_id)
        chunks.append(Result(page_content=doc, metadata=meta))
    return chunks


In [114]:
# Before prompt
# User query
def get_query(question):
    # Validation
    if not question:
        return "No question found."

    
    if isinstance(question, Item):
        question_text = question.title
    elif isinstance(question, dict):
        question_text = question.get('title', '')
    else:
        question_text = str(question).strip()

    if not question_text:
        return "No question found."

    return question_text

In [115]:
print(get_query("iphone 13 pro max 256gb"))   
print(type(get_query("iphone 13 pro max 256gb"))) 

iphone 13 pro max 256gb
<class 'str'>


In [116]:
test_item = items[1]
question_text = get_query(test_item)

In [117]:
question_text

'Duhome Faux Leather Vanity Chair, Swivel Makeup Stool Living Room Chair Upholstered Round Ottoman Stool Chair for Bedroom Entryway, Yellowish Brown'

In [118]:
# Query rewriting
@retry(wait=wait, stop=stop_after_attempt(3), reraise=True)
def rewrite_query(question):
    message = f"""
You are a pricing assistant. You will search a knowledge base of product descriptions and prices.
You have a user's question about a product, and you need to formulate a short, focused query to find the most relevant product descriptions.


And this is the user's current question:
{question}

Respond only with a single, refined query that you will use to search the knowledge base.
It should be a VERY short, specific description of the product, including key features, brand, and category.
Do not mention pricing or dollar amounts – just the product details.
IMPORTANT: Respond ONLY with the knowledgebase query, nothing else.
"""
    response = completion(model=GENERATOR_MODEL, messages=[{"role": "system", "content": message}])
    return response.choices[0].message.content

In [119]:
query = rewrite_query(question_text)

In [120]:
query

'Duhome swivel faux leather vanity chair yellowish brown round ottoman stool bedroom'

In [121]:
# Query Expansion
@retry(wait=wait, stop=stop_after_attempt(3), reraise=True)
def fetch_context_unranked(question):
    # Query embedding
    query = openai.embeddings.create(model=embedding_model, input=[question]).data[0].embedding
    # We dive into vector database collection to get the relevant chunks based on query embedding 
    # Retriever to get top-k 
    results = collection.query(query_embeddings=[query], n_results=RETRIEVAL_K)
    # Unranked chunks
    chunks = []
    for result in zip(results["documents"][0], results["metadatas"][0]):
        chunks.append(Result(page_content=result[0], metadata=result[1]))
    return chunks

In [122]:
# Original question 
chunks1 = fetch_context_unranked(question_text)

In [123]:
chunks1

[Result(page_content='BSHOMGI Vanity Stool Chair Faux Fur with Storage Soft Footrest Stool Upholstered Ottoman with Metal Legs Furry Padded Seat, Modern Multifunctional Chairs for Makeup, Bedroom (Beige02)\n\nTitle: BSHOMGI Vanity Stool Chair Faux Fur with Storage Soft Footrest Stool Upholstered Ottoman with Metal Legs Furry Padded Seat, Modern Multifunctional Chairs for Makeup, Bedroom (Beige02)\n\nCategory: home and kitchen\n\nDescription: Title: Faux Fur Vanity Stool with Storage and Metal Legs  \nCategory: Furniture / Bedroom Accessories  \nBrand: N/A  \nDescription: A modern, plush faux fur vanity stool featuring storage, sturdy metal legs, and a padded seat for comfort.  \nDetails: Includes easy assembly, waterproof and easy-to-clean fabric, high-density sponge padding, non-slip gold legs, and a compact size suitable for small spaces.', metadata={'price': 49.99, 'sku_codes': 'BEDROOM BEIGE02 BSHOMGI CHAIR CHAIRS FOOTREST FURRY MAKEUP METAL MODERN MULTIFUNCTIONAL OTTOMAN PADDED ST

In [124]:
# Rewritten question
chunks2 = fetch_context_unranked(query)

In [125]:
chunks2

[Result(page_content='BSHOMGI Vanity Stool Chair Faux Fur with Storage Soft Footrest Stool Upholstered Ottoman with Metal Legs Furry Padded Seat, Modern Multifunctional Chairs for Makeup, Bedroom (Beige02)\n\nTitle: BSHOMGI Vanity Stool Chair Faux Fur with Storage Soft Footrest Stool Upholstered Ottoman with Metal Legs Furry Padded Seat, Modern Multifunctional Chairs for Makeup, Bedroom (Beige02)\n\nCategory: home and kitchen\n\nDescription: Title: Faux Fur Vanity Stool with Storage and Metal Legs  \nCategory: Furniture / Bedroom Accessories  \nBrand: N/A  \nDescription: A modern, plush faux fur vanity stool featuring storage, sturdy metal legs, and a padded seat for comfort.  \nDetails: Includes easy assembly, waterproof and easy-to-clean fabric, high-density sponge padding, non-slip gold legs, and a compact size suitable for small spaces.', metadata={'source': 'product_18272', 'price': 49.99, 'sku_codes': 'BEDROOM BEIGE02 BSHOMGI CHAIR CHAIRS FOOTREST FURRY MAKEUP METAL MODERN MULTIF

In [126]:
# Merge chunks
def merge_chunks(chunks1, chunks2):
    merged = chunks1[:]
    existing = [chunk.page_content for chunk in chunks1]
    for chunk in chunks2:
        if chunk.page_content not in existing:
            merged.append(chunk)
    return merged

In [127]:
chunks = merge_chunks(chunks1, chunks2)

In [128]:
chunks

[Result(page_content='BSHOMGI Vanity Stool Chair Faux Fur with Storage Soft Footrest Stool Upholstered Ottoman with Metal Legs Furry Padded Seat, Modern Multifunctional Chairs for Makeup, Bedroom (Beige02)\n\nTitle: BSHOMGI Vanity Stool Chair Faux Fur with Storage Soft Footrest Stool Upholstered Ottoman with Metal Legs Furry Padded Seat, Modern Multifunctional Chairs for Makeup, Bedroom (Beige02)\n\nCategory: home and kitchen\n\nDescription: Title: Faux Fur Vanity Stool with Storage and Metal Legs  \nCategory: Furniture / Bedroom Accessories  \nBrand: N/A  \nDescription: A modern, plush faux fur vanity stool featuring storage, sturdy metal legs, and a padded seat for comfort.  \nDetails: Includes easy assembly, waterproof and easy-to-clean fabric, high-density sponge padding, non-slip gold legs, and a compact size suitable for small spaces.', metadata={'price': 49.99, 'sku_codes': 'BEDROOM BEIGE02 BSHOMGI CHAIR CHAIRS FOOTREST FURRY MAKEUP METAL MODERN MULTIFUNCTIONAL OTTOMAN PADDED ST

In [129]:
print(f"Chunks 1: {len(chunks1)}, Chunks 2: {len(chunks2)}, Merged Chunks: {len(chunks)}")

Chunks 1: 10, Chunks 2: 10, Merged Chunks: 12


In [130]:
# Reranking
# Rerank using LLM after retrival to improve the order of chunks based on relevance to the question
# Reranking
# Rerank using LLM after retrival to improve the order of chunks based on relevance to the question
@retry(wait=wait, stop=stop_after_attempt(3), reraise=True)
def rerank(question, chunks):
    system_prompt = """
You are a document re-ranker.
You are provided with a question and a list of relevant chunks of text from a query of a knowledge base.
The chunks are provided in the order they were retrieved; this should be approximately ordered by relevance, but you may be able to improve on that.
You must rank order the provided chunks by relevance to the question, with the most relevant chunk first.
Reply only with the list of ranked chunk ids, nothing else. Include all the chunk ids you are provided with, reranked.
"""
    user_prompt = f"The user has asked the following question:\n\n{question}\n\nOrder all the chunks of text by relevance to the question, from most relevant to least relevant. Include all the chunk ids you are provided with, reranked.\n\n"
    user_prompt += "Here are the chunks:\n\n"
    # Retrieve chunks and ids
    for index, chunk in enumerate(chunks):
        # We need to get text from each chunk to chove it with prompt
        user_prompt += f"# CHUNK ID: {index + 1}:\n\n{chunk.page_content}\n\n"
    user_prompt += "Reply only with the list of ranked chunk ids, nothing else."
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    response = completion(model=GENERATOR_MODEL, messages=messages, response_format=RankOrder)
    reply = response.choices[0].message.content
    order = RankOrder.model_validate_json(reply).order
    # Return the relavant chunks based on id in order

    # Defensive cleanup: the model can return a malformed order (missing ids,
    # duplicate ids, or out-of-range ids). Rather than letting that raise and
    # burn a retry attempt, keep only valid/unique ids from the model's order,
    # then append any leftover chunk ids (in original order) it dropped, so
    # no chunk is ever silently lost.
    valid_ids = set(range(1, len(chunks) + 1))
    seen = set()
    clean_order = []
    for i in order:
        if i in valid_ids and i not in seen:
            clean_order.append(i)
            seen.add(i)
    for i in valid_ids - seen:
        clean_order.append(i)

    return [chunks[i - 1] for i in clean_order]

In [131]:
reranked = rerank(question_text, chunks)

In [132]:
reranked = reranked[:RETRIEVAL_S]

In [133]:
len(reranked)

10

In [134]:
def fetch_context(original_question, rewritten_question):
    chunks1 = fetch_context_unranked(original_question)
    chunks2 = fetch_context_unranked(rewritten_question)
    chunks = merge_chunks(chunks1, chunks2)
    reranked = rerank(original_question, chunks)
    return reranked[:RETRIEVAL_S]

In [135]:
# Redefine fetch_context to use hybrid (dense+BM25) retrieval on each side,
def fetch_context(original_question, rewritten_question):
    chunks1 = fetch_context_hybrid(original_question)
    chunks2 = fetch_context_hybrid(rewritten_question)
    chunks = merge_chunks(chunks1, chunks2)
    reranked = rerank(original_question, chunks)
    return reranked[:RETRIEVAL_S]

In [136]:
chunks = fetch_context(question_text, query)

In [137]:
chunks

[Result(page_content='HULALA HOME Faux Leather Swivel Recliner Chair with Adjustable Backrest & Footrest, Modern Push Back Glider Recliner Rocking Chair, Home Theater Lounge Sofa Armchair for Living Room, Camel\n\nTitle: HULALA HOME Faux Leather Swivel Recliner Chair with Adjustable Backrest & Footrest, Modern Push Back Glider Recliner Rocking Chair, Home Theater Lounge Sofa Armchair for Living Room, Camel\n\nCategory: home and kitchen\n\nDescription: Title: HULALA HOME Faux Leather Swivel Recliner with Adjustable Backrest and Footrest  \nCategory: Furniture  \nBrand: HULALA HOME  \nDescription: Modern faux leather swivel recliner offering customizable comfort with adjustable backrest and footrest.  \nDetails: Features a 360-degree swivel, reclining from 105° to 135°, cushioned support, sturdy construction, and easy assembly.', metadata={'source': 'product_15143', 'price': 469.99, 'title': 'HULALA HOME Faux Leather Swivel Recliner Chair with Adjustable Backrest & Footrest, Modern Push 

In [138]:
len(chunks)

10

In [139]:
# After prompt
# Create prompt
def make_rag_messages(question, chunks):
    # Context is most of meta data
    context = "\n\n".join(
        f"""Product:
    Title: {chunk.metadata['title']}
    Category: {chunk.metadata['type']}
    Price: ${chunk.metadata['price']}
    Description:{chunk.page_content}
    Source: {chunk.metadata['source']}"""
            for chunk in chunks
        )
    # Shoving in system prompt with context
    system_prompt = SYSTEM_PROMPT.format(context=context)
    # Question is user prompt
    return [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": question}]

In [140]:
messages = make_rag_messages(question_text, chunks)

In [141]:
display(Markdown(messages[0]["content"]))


You are a pricing assistant for an e-commerce platform.

Your task is to estimate the price of the target product using
the retrieved products and their prices as reference examples.

Use the retrieved products that are most similar to the target product,
even if they are not an exact match - same category, similar brand tier,
similar features/size are enough to use as comparables. Reason about how
the target differs (bigger/smaller, more/fewer features, premium/budget
brand) and adjust your estimate accordingly.

Only return 0.00 if the retrieved context is from a clearly different
category or is otherwise unusable for comparison - not merely because no
retrieved product is an exact match.

Return ONLY one numeric value representing the estimated price
in US dollars. Do not include currency symbols, explanations, or
additional text.

Knowledge base context:
Product:
    Title: HULALA HOME Faux Leather Swivel Recliner Chair with Adjustable Backrest & Footrest, Modern Push Back Glider Recliner Rocking Chair, Home Theater Lounge Sofa Armchair for Living Room, Camel
    Category: home and kitchen
    Price: $469.99
    Description:HULALA HOME Faux Leather Swivel Recliner Chair with Adjustable Backrest & Footrest, Modern Push Back Glider Recliner Rocking Chair, Home Theater Lounge Sofa Armchair for Living Room, Camel

Title: HULALA HOME Faux Leather Swivel Recliner Chair with Adjustable Backrest & Footrest, Modern Push Back Glider Recliner Rocking Chair, Home Theater Lounge Sofa Armchair for Living Room, Camel

Category: home and kitchen

Description: Title: HULALA HOME Faux Leather Swivel Recliner with Adjustable Backrest and Footrest  
Category: Furniture  
Brand: HULALA HOME  
Description: Modern faux leather swivel recliner offering customizable comfort with adjustable backrest and footrest.  
Details: Features a 360-degree swivel, reclining from 105° to 135°, cushioned support, sturdy construction, and easy assembly.
    Source: product_15143

Product:
    Title: Olela Faux Leather Accent Chair with Arms for Living Room, Modern Tufted Single Sofa Armchair with Gold Metal Legs Upholstered Reading Chair for Bedroom Office Decorative (Brown - Leather)
    Category: home and kitchen
    Price: $209.99
    Description:Olela Faux Leather Accent Chair with Arms for Living Room, Modern Tufted Single Sofa Armchair with Gold Metal Legs Upholstered Reading Chair for Bedroom Office Decorative (Brown - Leather)

Title: Olela Faux Leather Accent Chair with Arms for Living Room, Modern Tufted Single Sofa Armchair with Gold Metal Legs Upholstered Reading Chair for Bedroom Office Decorative (Brown - Leather)

Category: home and kitchen

Description: Title: Olela Faux Leather Accent Armchair with Gold Legs  
Category: Furniture  
Brand: Olela  
Description: Modern tufted accent chair upholstered in faux leather with stylish gold metal legs, perfect for versatile spaces.  
Details: Features a plush foam cushion, ergonomic design, easy assembly, and sturdy metal construction, supporting up to 220 lbs.
    Source: product_15975

Product:
    Title: Homebeez Modern Upholstered Entryway Cushion Hallway Metal Bedroom Bench Ottoman for Living Room Bedroom, 46.9" Length, Brown
    Category: home and kitchen
    Price: $92.99
    Description:Homebeez Modern Upholstered Entryway Cushion Hallway Metal Bedroom Bench Ottoman for Living Room Bedroom, 46.9" Length, Brown

Title: Homebeez Modern Upholstered Entryway Cushion Hallway Metal Bedroom Bench Ottoman for Living Room Bedroom, 46.9" Length, Brown

Category: home and kitchen

Description: Title: Homebeez Modern Upholstered Entryway Bench  
Category: Furniture  
Brand: Homebeez  
Description: Stylish and sturdy brown upholstered hall bench perfect for entryways, bedrooms, and living rooms.  
Details: Features eye-catching cane accents, high-quality faux leather and metal construction, supports up to 300 lbs, and easy assembly with included hardware and tools.
    Source: product_17515

Product:
    Title: Safeplus Folding Storage Ottoman Bench, 45" Large Ottoman Space-Saving Bench,Faux Leather Storage Chest with Memory Foam,Footrest Padded Stool Seat for Bedroom Toy Chest, Black,Leather
    Category: home and kitchen
    Price: $64.99
    Description:Safeplus Folding Storage Ottoman Bench, 45" Large Ottoman Space-Saving Bench,Faux Leather Storage Chest with Memory Foam,Footrest Padded Stool Seat for Bedroom Toy Chest, Black,Leather

Title: Safeplus Folding Storage Ottoman Bench, 45" Large Ottoman Space-Saving Bench,Faux Leather Storage Chest with Memory Foam,Footrest Padded Stool Seat for Bedroom Toy Chest, Black,Leather

Category: home and kitchen

Description: Title: Safeplus 45-Inch Folding Storage Ottoman Bench  
Category: Furniture  
Brand: Safeplus  
Description: Spacious, stylish, and versatile storage ottoman with seating and footrest functions.  
Details: Crafted from faux leather with memory foam top, supports up to 660 lbs, folds for easy storage, and features a removable lid for organized space.
    Source: product_15542

Product:
    Title: BSHOMGI Vanity Stool Chair Faux Fur with Storage Soft Footrest Stool Upholstered Ottoman with Metal Legs Furry Padded Seat, Modern Multifunctional Chairs for Makeup, Bedroom (Beige02)
    Category: home and kitchen
    Price: $49.99
    Description:BSHOMGI Vanity Stool Chair Faux Fur with Storage Soft Footrest Stool Upholstered Ottoman with Metal Legs Furry Padded Seat, Modern Multifunctional Chairs for Makeup, Bedroom (Beige02)

Title: BSHOMGI Vanity Stool Chair Faux Fur with Storage Soft Footrest Stool Upholstered Ottoman with Metal Legs Furry Padded Seat, Modern Multifunctional Chairs for Makeup, Bedroom (Beige02)

Category: home and kitchen

Description: Title: Faux Fur Vanity Stool with Storage and Metal Legs  
Category: Furniture / Bedroom Accessories  
Brand: N/A  
Description: A modern, plush faux fur vanity stool featuring storage, sturdy metal legs, and a padded seat for comfort.  
Details: Includes easy assembly, waterproof and easy-to-clean fabric, high-density sponge padding, non-slip gold legs, and a compact size suitable for small spaces.
    Source: product_18272

Product:
    Title: BNEHS Recliner Chair, Faux Leather Recliner Sofa, Classical Reclining Chair w/8 Massage Positions and Lumbar Heating, 360 Degree Swivel, Beige
    Category: home and kitchen
    Price: $359.99
    Description:BNEHS Recliner Chair, Faux Leather Recliner Sofa, Classical Reclining Chair w/8 Massage Positions and Lumbar Heating, 360 Degree Swivel, Beige

Title: BNEHS Recliner Chair, Faux Leather Recliner Sofa, Classical Reclining Chair w/8 Massage Positions and Lumbar Heating, 360 Degree Swivel, Beige

Category: home and kitchen

Description: Title: BNEHS Classic Faux Leather Swivel Recliner with Massage & Heating  
Category: Furniture  
Brand: BNEHS  
Description: Elegant beige recliner chair featuring 8 massage modes, lumbar heating, 360° swivel, and breathable faux leather for ultimate relaxation.  
Details: Includes rocking and swiveling functions, remote-controlled massage and heating, side storage pockets, and comfort-enhancing design for living room or bedroom use.
    Source: product_14731

Product:
    Title: Joveco Small Ottoman Footrest Stool Linen Square Footstool Modern Soft Padded Seat for Living Room Bedroom Couch Chair Dogs (Beige)
    Category: home and kitchen
    Price: $42.99
    Description:Joveco Small Ottoman Footrest Stool Linen Square Footstool Modern Soft Padded Seat for Living Room Bedroom Couch Chair Dogs (Beige)

Title: Joveco Small Ottoman Footrest Stool Linen Square Footstool Modern Soft Padded Seat for Living Room Bedroom Couch Chair Dogs (Beige)

Category: home and kitchen

Description: Title: Joveco Small Modern Linen Square Ottoman  
Category: Furniture  
Brand: Joveco  
Description: A lightweight and stylish beige linen ottoman, perfect for various home and office spaces.  
Details: Features a wooden frame, padded linen seat, easy assembly with removable legs, and additional storage space underneath.
    Source: product_16783

Product:
    Title: Small foot stool ottoman, Teal Velvet rectangle ottoman footrest, bedside step stool with wood legs, small Rectangular stool, foot rest for couch, small leather ottoman for desk, living room, bedroom
    Category: home and kitchen
    Price: $38.99
    Description:Small foot stool ottoman, Teal Velvet rectangle ottoman footrest, bedside step stool with wood legs, small Rectangular stool, foot rest for couch, small leather ottoman for desk, living room, bedroom

Title: Small foot stool ottoman, Teal Velvet rectangle ottoman footrest, bedside step stool with wood legs, small Rectangular stool, foot rest for couch, small leather ottoman for desk, living room, bedroom

Category: home and kitchen

Description: Title: Teal Velvet Small Rectangular Ottoman Footrest
Category: Home Furniture
Brand: DURFII
Description: A versatile, compact teal velvet ottoman designed for footrest, seating, and small step stool needs.
Details: Features a sturdy wood frame, waterproof velvet cover, lightweight portability with handle, and easy assembly for use in living rooms, bedrooms, or beside desks.
    Source: product_16479

Product:
    Title: Poundex PDEX-F1144 2 Piece 1Perfect Choice Dining High Counter Height Side Chair Bar Stool, Faux Leather, Expresso
    Category: home and kitchen
    Price: $199.0
    Description:Poundex PDEX-F1144 2 Piece 1Perfect Choice Dining High Counter Height Side Chair Bar Stool, Faux Leather, Expresso

Title: Poundex PDEX-F1144 2 Piece 1Perfect Choice Dining High Counter Height Side Chair Bar Stool, Faux Leather, Expresso

Category: home and kitchen

Description: Title: Poundex Espresso Faux Leather Counter Height Bar Stools (Set of 2)  
Category: Furniture  
Brand: Poundex  
Description: Elegant high-back counter height chairs with faux leather upholstery and wooden legs, ideal for modern dining or bar areas.  
Details: Features a 24-inch seat height, tufted backrest, and a stylish Parsons design in espresso finish.
    Source: product_17419

Product:
    Title: Upholstered Adjustable Bar Stool Ecru and Chrome 130503
    Category: home and kitchen
    Price: $142.22
    Description:Upholstered Adjustable Bar Stool Ecru and Chrome 130503

Title: Upholstered Adjustable Bar Stool Ecru and Chrome 130503

Category: home and kitchen

Description: Title: Ecru and Chrome Adjustable Bar Stool  
Category: Furniture  
Brand: Coaster Home Furnishings  
Description: Stylish and comfortable adjustable bar stool suitable for entertainment, home office, or professional settings.  
Details: Features a padded Ecru leatherette seat with a sleek chrome base, 360° swivel, pneumatic height adjustment, and a durable engineered wood frame.
    Source: product_14057


In [142]:
def get_answer(messages):
    response = completion(model=GENERATOR_MODEL, messages=messages)
    return response.choices[0].message.content

In [143]:
response = get_answer(messages)

In [144]:
response

'190.00'

In [145]:
# Answer function
def answer_question(question: str) -> tuple[str, list]:
    question_text = get_query(question)
    # Query rewriting
    query = rewrite_query(question_text)
    print(query)
    # Before prompt
    chunks = fetch_context(question_text, query)
    # After prompt
    messages = make_rag_messages(question_text, chunks)
    response = get_answer(messages)
    # Return string which is response and list of chunks that we retrieved answer from
    return response, chunks

In [146]:
# Structured price-estimate output for agentic use
from typing import Literal

class PriceEstimate(BaseModel):
    """Structured result of a price estimation - safe for an agent to branch on."""

    estimated_price: float | None = Field(
        default=None,
        description="Estimated price in USD. Must be null (not 0.0) if no reasonable estimate is possible.",
    )
    confidence: Literal["high", "medium", "low", "none"] = Field(
        description="'none' means the retrieved context could not support any estimate "
                    "(e.g. wrong category / no comparable products) - an agent should treat "
                    "this as 'need another source', not as a $0 product."
    )
    matched_category: bool = Field(
        description="Whether the retrieved context was from the same product category as the target"
    )
    comparable_products_used: int = Field(
        description="Number of retrieved products actually used as comparables for the estimate"
    )
    reasoning: str = Field(description="1-2 sentence explanation of how the estimate was derived")


STRUCTURED_SYSTEM_PROMPT = """
You are a pricing assistant for an e-commerce platform.

Estimate the price of the target product using the retrieved products and their
prices as reference examples. Use products that are similar even if not an exact
match (same category, similar brand tier/features/size are enough) and reason
about how the target differs to adjust your estimate.

Set confidence to:
- "high": very similar comparables found, price estimate should be reliable
- "medium": same-category comparables found but with notable differences
- "low": only loosely related comparables, estimate is a rough guess
- "none": retrieved context is a different category or otherwise unusable -
  in this case estimated_price MUST be null, not 0.0

Knowledge base context:
{context}
"""


def get_structured_answer(question: str, chunks: list[Result]) -> PriceEstimate:
    context = "\n\n".join(
        f"""Product:
    Title: {chunk.metadata['title']}
    Category: {chunk.metadata['type']}
    Price: ${chunk.metadata['price']}
    Description:{chunk.page_content}"""
        for chunk in chunks
    )
    messages = [
        {"role": "system", "content": STRUCTURED_SYSTEM_PROMPT.format(context=context)},
        {"role": "user", "content": question},
    ]
    response = completion(model=GENERATOR_MODEL, messages=messages, response_format=PriceEstimate)
    return PriceEstimate.model_validate_json(response.choices[0].message.content)


def answer_question(question: str) -> tuple[PriceEstimate, list[Result]]:
    question_text = get_query(question)
    query = rewrite_query(question_text)
    chunks = fetch_context(question_text, query)
    estimate = get_structured_answer(question_text, chunks)
    return estimate.estimated_price, chunks


In [147]:
test_item = items[2]
answer, retrieved_chunks = answer_question(test_item)
print("Actual price:", test_item.price)
print("Predicted price:", answer)

Actual price: 9.99
Predicted price: 16.99


In [148]:
# Try normal question
answer_question("iphone 13 pro max 256gb")

(950.0,
 [Result(page_content='ULANZI Super Suction Power Holder - for Tablet and Cell Phones, iPhone 13 Pro Max, iPad Pro 12.9/9.7, iPad, iPad Air 2, iPad Mini, Surface Pro Series, and Samsung Galaxy Suitable for Tab and Fire\n\nTitle: ULANZI Super Suction Power Holder - for Tablet and Cell Phones, iPhone 13 Pro Max, iPad Pro 12.9/9.7, iPad, iPad Air 2, iPad Mini, Surface Pro Series, and Samsung Galaxy Suitable for Tab and Fire\n\nCategory: electronics\n\nDescription: Title: ULANZI Super Suction Power Holder for Tablets and Phones  \nCategory: Electronics  \nBrand: ULANZI  \nDescription: A powerful electronic vacuum suction holder compatible with various tablets and smartphones.  \nDetails: Features a 360-degree rotatable support plate, adjustable angles and height, high stability with aluminum and steel construction, and easy tool-free assembly.', metadata={'product_id': 5811, 'type': 'electronics', 'sku_codes': 'GALAXY HOLDER IPHONE PHONES POWER SAMSUNG SERIES SUCTION SUITABLE SUPER